# 05 - Model Explainability

This notebook explains the v3 AQI forecasting model using global feature importance, SHAP values, and error analysis by city/month.

In [ ]:
from pathlib import Path
import json
import warnings

import joblib
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
from IPython.display import display

sns.set_theme(style='whitegrid')

TEST_PATH = Path('data/splits/test_v3.csv')
MODEL_PATH = Path('outputs/models/best_model_v3.pkl')
FEATURES_PATH = Path('outputs/models/feature_columns_v3.json')

REPORTS_DIR = Path('outputs/reports')
FIG_DIR = Path('outputs/figures')
REPORTS_DIR.mkdir(parents=True, exist_ok=True)
FIG_DIR.mkdir(parents=True, exist_ok=True)


## Step 1 - Load Data, Model, And Recreate Predictions

Forecasting v3 saves the best model and metrics, not a separate prediction CSV. Predictions are reproduced here from the saved model and test split.

In [ ]:
for path in [TEST_PATH, MODEL_PATH, FEATURES_PATH]:
    if not path.exists():
        raise FileNotFoundError(f'Missing required artifact: {path}')

test_df = pd.read_csv(TEST_PATH)
feature_columns = json.loads(FEATURES_PATH.read_text(encoding='utf-8'))
model = joblib.load(MODEL_PATH)

required_cols = ['Timestamp', 'target_timestamp', 'City', 'AQI'] + feature_columns
missing = [col for col in required_cols if col not in test_df.columns]
if missing:
    raise ValueError(f'Missing columns in test_v3: {missing}')

test_df['Timestamp'] = pd.to_datetime(test_df['Timestamp'])
test_df['target_timestamp'] = pd.to_datetime(test_df['target_timestamp'])
X_test = test_df[feature_columns].astype(float)
y_test = test_df['AQI'].astype(float)

if isinstance(model, dict) and model.get('type') == 'naive_baseline':
    raise ValueError('Explainability requires a trained estimator, but best_model_v3 is a naive baseline dictionary.')

predictions = model.predict(X_test)
eval_df = test_df[['Timestamp', 'target_timestamp', 'City']].copy()
eval_df['Actual_AQI'] = y_test.values
eval_df['Predicted_AQI'] = predictions
eval_df['Error'] = eval_df['Actual_AQI'] - eval_df['Predicted_AQI']
eval_df['Absolute_Error'] = eval_df['Error'].abs()
eval_df['Month'] = eval_df['Timestamp'].dt.month

print('Test rows:', len(test_df))
print('Feature count:', len(feature_columns))
print('Model type:', type(model).__name__)
display(eval_df.head())


## Step 2 - Global Feature Importance

In [ ]:
if not hasattr(model, 'feature_importances_'):
    raise AttributeError(f'{type(model).__name__} does not expose feature_importances_.')

feature_importance = (
    pd.DataFrame({'feature': feature_columns, 'importance': model.feature_importances_})
    .sort_values('importance', ascending=False)
    .reset_index(drop=True)
)
feature_importance.to_csv(REPORTS_DIR / 'feature_importance.csv', index=False)
top10_feature_importance = feature_importance.head(10)
top10_feature_importance.to_csv(REPORTS_DIR / 'top10_feature_importance.csv', index=False)

top15 = feature_importance.head(15)
plt.figure(figsize=(12, 8))
sns.barplot(data=top15, x='importance', y='feature', color='steelblue')
plt.title('Top 15 Feature Importances - AQI Forecasting Model')
plt.xlabel('Importance')
plt.ylabel('Feature')
plt.tight_layout()
plt.savefig(FIG_DIR / 'feature_importance.png', dpi=200)
plt.close()

display(top15)
print('Saved:', REPORTS_DIR / 'feature_importance.csv')
print('Saved:', REPORTS_DIR / 'top10_feature_importance.csv')
print('Saved:', FIG_DIR / 'feature_importance.png')


## Step 3 - Local Explanation With SHAP

A 100-row sample is used to keep SHAP execution fast and readable.

In [ ]:
import shap

sample_size = min(100, len(X_test))
X_sample = X_test.sample(n=sample_size, random_state=42)

explainer = shap.TreeExplainer(model)
shap_values = explainer.shap_values(X_sample)

plt.figure()
shap.summary_plot(shap_values, X_sample, show=False, max_display=15)
plt.tight_layout()
plt.savefig(FIG_DIR / 'shap_summary.png', dpi=200, bbox_inches='tight')
plt.close()

print('Saved:', FIG_DIR / 'shap_summary.png')


## Optional Advanced - SHAP Force Plot For One Prediction

In [ ]:
force_plot_path = FIG_DIR / 'shap_force_plot.html'
with warnings.catch_warnings():
    warnings.simplefilter('ignore')
    force_plot = shap.force_plot(
        explainer.expected_value,
        shap_values[0],
        X_sample.iloc[0],
        matplotlib=False,
    )
shap.save_html(str(force_plot_path), force_plot)
print('Saved:', force_plot_path)


## Step 4 - Error Analysis

In [ ]:
error_by_city = (
    eval_df.groupby('City', as_index=False)['Absolute_Error']
    .mean()
    .rename(columns={'Absolute_Error': 'MAE'})
    .sort_values('MAE', ascending=False)
)

plt.figure(figsize=(12, 6))
sns.barplot(data=error_by_city, x='City', y='MAE', color='steelblue')
plt.title('Mean Absolute Error by City')
plt.xlabel('City')
plt.ylabel('MAE')
plt.xticks(rotation=25)
plt.tight_layout()
plt.savefig(FIG_DIR / 'error_by_city.png', dpi=200)
plt.close()

error_by_month = (
    eval_df.groupby('Month', as_index=False)['Absolute_Error']
    .mean()
    .rename(columns={'Absolute_Error': 'MAE'})
)

plt.figure(figsize=(12, 6))
sns.lineplot(data=error_by_month, x='Month', y='MAE', marker='o')
plt.title('Mean Absolute Error by Month')
plt.xlabel('Month')
plt.ylabel('MAE')
plt.xticks(range(1, 13))
plt.tight_layout()
plt.savefig(FIG_DIR / 'error_by_month.png', dpi=200)
plt.close()

display(error_by_city)
display(error_by_month)
print('Saved:', FIG_DIR / 'error_by_city.png')
print('Saved:', FIG_DIR / 'error_by_month.png')


## Error By Hotspot Cluster

This joins Phase 4 cluster labels to forecast errors to check whether severe hotspot clusters are harder to predict.

In [ ]:
clusters_path = REPORTS_DIR / 'city_clusters.csv'
if clusters_path.exists():
    cluster_df = pd.read_csv(clusters_path)[['City', 'Cluster', 'Cluster_Name']]
    eval_with_clusters = eval_df.merge(cluster_df, on='City', how='left')
    error_by_cluster = (
        eval_with_clusters.groupby(['Cluster', 'Cluster_Name'], as_index=False)['Absolute_Error']
        .mean()
        .rename(columns={'Absolute_Error': 'MAE'})
        .sort_values('MAE', ascending=False)
    )
    error_by_cluster.to_csv(REPORTS_DIR / 'error_by_cluster.csv', index=False)

    plt.figure(figsize=(10, 6))
    sns.barplot(data=error_by_cluster, x='Cluster_Name', y='MAE', color='steelblue')
    plt.title('Mean Absolute Error by Hotspot Cluster')
    plt.xlabel('Cluster')
    plt.ylabel('MAE')
    plt.xticks(rotation=20, ha='right')
    plt.tight_layout()
    plt.savefig(FIG_DIR / 'error_by_cluster.png', dpi=200)
    plt.close()

    display(error_by_cluster)
    print('Saved:', REPORTS_DIR / 'error_by_cluster.csv')
    print('Saved:', FIG_DIR / 'error_by_cluster.png')
else:
    error_by_cluster = pd.DataFrame()
    print('city_clusters.csv not found; skipped error-by-cluster analysis.')


## Step 5 - Observations

- The most important features indicate whether the model relies mainly on current pollutant levels, lag history, rolling pollutant behavior, or weather inputs.
- Weather influence can be assessed by checking the rank and importance of wind, precipitation, temperature, and humidity features.
- Cities with higher MAE are harder to predict and may have stronger episodic pollution spikes or noisier source patterns.
- Monthly error trends reveal whether the model struggles more during seasonal transitions, winter pollution episodes, or monsoon periods.
- SHAP analysis shows PM2.5 increases AQI, while wind speed and precipitation reduce it, aligning with known physical effects.

In [ ]:
top_features = feature_importance.head(5)['feature'].tolist()
weather_features = [f for f in feature_importance['feature'] if any(key in f for key in ['wind', 'precip', 'temperature', 'humidity'])]
hardest_city = error_by_city.iloc[0]
hardest_month = error_by_month.sort_values('MAE', ascending=False).iloc[0]
hardest_cluster = error_by_cluster.iloc[0] if not error_by_cluster.empty else None

print('Key computed observations:')
print(f"- Top 5 features: {', '.join(top_features)}")
print(f"- Highest-ranked weather feature: {weather_features[0] if weather_features else 'None'}")
print(f"- Hardest city to predict: {hardest_city['City']} (MAE={hardest_city['MAE']:.2f})")
print(f"- Highest-error month: {int(hardest_month['Month'])} (MAE={hardest_month['MAE']:.2f})")
if hardest_cluster is not None:
    print(f"- Highest-error cluster: {hardest_cluster['Cluster_Name']} (MAE={hardest_cluster['MAE']:.2f})")
print('- Directional SHAP insight: PM2.5 increases AQI, while wind speed and precipitation reduce it, matching known dispersion and washout effects.')
